<a href="https://colab.research.google.com/github/Areefbaba/Insurance-AI-Predictor/blob/main/model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error,mean_squared_error, r2_score

In [29]:
df=pd.read_csv("insurance.csv")

In [30]:
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


In [32]:
df.isnull().sum()

,0
age,0
sex,0
bmi,0
children,0
smoker,0
region,0
charges,0


In [33]:
df.describe()

,age,bmi,children,charges
count,1338.000000,1338.000000,1338.000000,1338.000000
mean,39.207025,30.663397,1.094918,13270.422265
std,14.049960,6.098187,1.205493,12110.011237
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.296250,0.000000,4740.287150
50%,39.000000,30.400000,1.000000,9382.033000
75%,51.000000,34.693750,2.000000,16639.912515
max,64.000000,53.130000,5.000000,63770.428010


In [34]:

df['age'] = df['age'].fillna(df['age'].mean())
df['bmi'] = df['bmi'].fillna(df['bmi'].mean())
df['children'] = df['children'].fillna(df['children'].mean())

df['sex'] = df['sex'].fillna(df['sex'].mode()[0])
df['smoker'] = df['smoker'].fillna(df['smoker'].mode()[0])
df['region'] = df['region'].fillna(df['region'].mode()[0])

In [35]:
df['sex'] = df['sex'].map({ 'male':1,'female':0})
df['smoker'] = df['smoker'].map({'yes':1,'no':0})

In [36]:
df = pd.get_dummies(df,columns=['region'],drop_first=True)
df.head()

,age,sex,bmi,children,smoker,charges,region_northwest,region_southeast,region_southwest
0,19,0,27.900,0,1,16884.92400,False,False,True
1,18,1,33.770,1,0,1725.55230,False,True,False
2,28,1,33.000,3,0,4449.46200,False,True,False
3,33,1,22.705,0,0,21984.47061,True,False,False
4,32,1,28.880,0,0,3866.85520,True,False,False


In [37]:
x = df.drop('charges', axis=1)
y = df['charges']

In [38]:
x.columns

Index(['age', 'sex', 'bmi', 'children', 'smoker', 'region_northwest',
       'region_southeast', 'region_southwest'],
      dtype='object')

In [39]:
x_train, x_test,y_train, y_test = train_test_split( x,y, test_size=0.2,random_state=42)

In [40]:
model = LinearRegression()
model.fit(x_train, y_train)

LinearRegression()

In [41]:
y_pred = model.predict(x_test)
y_pred[:5]

array([ 8969.55027444,  7068.74744287, 36858.41091155,  9454.67850053,
       26973.17345656])

In [42]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
print("MAE :", mae)
print("MSE :", mse)
print("RMSE :", rmse)
print("R2 Score :", r2)

MAE : 4181.1944737536505
MSE : 33596915.85136146
RMSE : 5796.2846592762735
R2 Score : 0.7835929767120723


In [43]:
results = pd.DataFrame({
    'Actual': y_test,
    'Predicted': y_pred
})

results.head(10)

,Actual,Predicted
764,9095.06825,8969.550274
887,5272.17580,7068.747443
890,29330.98315,36858.410912
1293,9301.89355,9454.678501
259,33750.29180,26973.173457
1312,4536.25900,10864.113164
899,2117.33885,170.280841
752,14210.53595,16903.450287
1286,3732.62510,1092.430936
707,10264.44210,11218.343184


In [44]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import LinearSVR
from xgboost import XGBRegressor
models = {

    'XGBoost Regressor': XGBRegressor(
        n_estimators=1000,
        learning_rate=0.01,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    ),
    'Decision Tree': DecisionTreeRegressor(),
    'Random Forest': RandomForestRegressor(),
    'AdaBoost': AdaBoostRegressor(),
    'Gradient Boosting': GradientBoostingRegressor(),
    'KNeighbors': KNeighborsRegressor(),
    'LinearSVR': LinearSVR()
}

models
from sklearn.metrics import mean_squared_error as mse,r2_score as r2

In [45]:
result={}
for name,model in models.items():
  model.fit(x_train,y_train)
  y_pred=model.predict(x_test)
  MSE=mse(y_test,y_pred)
  R2=r2(y_test,y_pred)
  print(f"{name} R2:{R2}")
  print(f"{name} MSE:{MSE}")
  result[name]=R2
best_model = max(result, key=result.get)
print(f"BEST MODEL:{best_model}")

XGBoost Regressor R2:0.8802172159718673
XGBoost Regressor MSE:18596125.27492067
Decision Tree R2:0.6832685981344411
Decision Tree MSE:49172148.36323916
Random Forest R2:0.8611550929575725
Random Forest MSE:21555495.692430068
AdaBoost R2:0.834084011655065
AdaBoost MSE:25758246.724755034
Gradient Boosting R2:0.8792900291779596
Gradient Boosting MSE:18740069.848530754
KNeighbors R2:0.1869864737173137
KNeighbors MSE:126219318.64104068
LinearSVR R2:-0.07255616428890699
LinearSVR MSE:166512983.95953503
BEST MODEL:XGBoost Regressor


In [46]:
y_pred = models[best_model].predict(x_test)

In [49]:
def pred_model(model):
    age = int(input("Enter Age: "))
    sex_input = input("Sex (1=Male, 0=Female):  ").lower()
    bmi = float(input("Enter BMI: "))
    children = int(input("Enter Number of Children: "))
    smoker_input = input("Smoker? (1=Yes, 0=No): ").lower()
    print("Region:")
    print("100 = Northwest")
    print("010 = Southeast")
    print("001 = Southwest")
    print("000 = Northeast")

    region = input("Enter region code: ")

    # Convert region code into ML features
    region_northwest = int(region[0])
    region_southeast = int(region[1])
    region_southwest = int(region[2])

    # Convert sex and smoker to integers
    sex = int(sex_input)
    smoker = int(smoker_input)

    # Final input for model
    user_data = [[
        age,
        sex,
        bmi,
        children,
        smoker,
        region_northwest,
        region_southeast,
        region_southwest
    ]]
    # Prediction
    prediction = model.predict(user_data)
    print("\nPredicted Insurance Charges:", prediction[0])

pred_model(models[best_model])

Enter Age: 46
Sex (1=Male, 0=Female):  1
Enter BMI: 22.3
Enter Number of Children: 0
Smoker? (1=Yes, 0=No): 0
Region:
100 = Northwest
010 = Southeast
001 = Southwest
000 = Northeast
Enter region code: 001

Predicted Insurance Charges: 8865.786
